# Exploratory Analysis — Global Earthquake & Tsunami Events (2001–2022)

**Contents:**

- Load dataset and validate
- Basic data inspection
- Time-based analysis (yearly trends, tsunami counts)
- Magnitude & depth analysis
- Geographic distribution (latitude vs longitude)
- Correlation analysis
- Save key visuals to `/mnt/data/visuals/`

> This notebook was auto-generated. If you run into any issues, restart the kernel and run all cells.


In [ ]:
# Imports & plotting configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline

VIS_PATH = '/mnt/data/visuals'
os.makedirs(VIS_PATH, exist_ok=True)

print('Libraries loaded. Visuals will be saved to', VIS_PATH)


In [ ]:
# Load dataset (robust to common filename locations)
possible_paths = [
    '/mnt/data/earthquake_data_tsunami.csv',
    'earthquake_data_tsunami.csv',
    './data/earthquake_data_tsunami.csv',
    '../data/earthquake_data_tsunami.csv'
]

csv_path = None
for p in possible_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError('Could not find earthquake_data_tsunami.csv in expected locations.\nChecked: ' + str(possible_paths))

print('Using CSV:', csv_path)

df = pd.read_csv(csv_path)
print('Loaded dataframe with shape:', df.shape)

# Show columns
print('\nColumns:')
print(df.columns.tolist())

df.head()


In [ ]:
# Standardize column names (lowercase, strip)
orig_cols = df.columns.tolist()
new_cols = [c.strip().lower() for c in orig_cols]
rename_map = {o: n for o, n in zip(orig_cols, new_cols)}
df.rename(columns=rename_map, inplace=True)

# Common expected names and fallback mapping
fallbacks = {
    'time': ['time', 'datetime', 'date'],
    'latitude': ['latitude', 'lat'],
    'longitude': ['longitude', 'lon', 'long'],
    'depth': ['depth', 'depth_km', 'depth (km)'],
    'mag': ['mag', 'magnitude', 'mag_mw'],
    'place': ['place', 'location'],
    'tsunami': ['tsunami', 'tsunami_flag', 'tsunami?']
}

# Find actual column names
cols_found = {}
for key, options in fallbacks.items():
    for opt in options:
        if opt in df.columns:
            cols_found[key] = opt
            break

# If any required column missing, raise informative error
required = ['time', 'latitude', 'longitude', 'depth', 'mag', 'tsunami']
missing = [r for r in required if r not in cols_found]
if missing:
    print('Warning: Could not auto-detect these required columns:', missing)
    print('Detected columns:', list(df.columns))

# For convenience, assign aliases if found
for k, v in cols_found.items():
    df.rename(columns={v: k}, inplace=True)

print('\nColumns after mapping:')
print(df.columns.tolist())

# Quick type conversions
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df['year'] = df['time'].dt.year

# Convert tsunami to numeric if possible
if 'tsunami' in df.columns:
    try:
        df['tsunami'] = pd.to_numeric(df['tsunami'], errors='coerce').fillna(0).astype(int)
    except Exception:
        df['tsunami'] = df['tsunami'].astype('category').cat.codes

# Show info
print('\nDataframe info:')
df.info()

# Show missing values summary
print('\nMissing values per column:')
print(df.isnull().sum())


In [ ]:
# Basic statistics
print('Rows, Columns:', df.shape)

# Numeric describe
df.describe(include='all')


In [ ]:
# Yearly earthquake counts
if 'year' not in df.columns and 'time' in df.columns:
    df['year'] = pd.DatetimeIndex(df['time']).year

yearly = df['year'].value_counts().sort_index()

plt.figure(figsize=(12,5))
plt.plot(yearly.index, yearly.values, marker='o')
plt.title('Earthquake Occurrences per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(VIS_PATH, 'yearly_trend.png'))
plt.show()

# Tsunami vs Non-tsunami per year (stacked bars)
if 'tsunami' in df.columns:
    cros = df.groupby(['year', 'tsunami']).size().unstack(fill_value=0)
    cros = cros.sort_index()
    plt.figure(figsize=(12,6))
    bottom = np.zeros(len(cros))
    plt.bar(cros.index, cros.get(0, 0), label='No Tsunami')
    if 1 in cros.columns:
        plt.bar(cros.index, cros[1], bottom=cros.get(0,0), label='Tsunami')
    plt.title('Tsunami vs Non-Tsunami Earthquakes by Year')
    plt.xlabel('Year')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_PATH, 'tsunami_vs_non_tsunami_by_year.png'))
    plt.show()
else:
    print('Tsunami column not found; skipping tsunami yearly comparison.')


In [ ]:
# Magnitude distribution
if 'mag' in df.columns:
    plt.figure(figsize=(10,4))
    plt.hist(df['mag'].dropna(), bins=30)
    plt.title('Distribution of Earthquake Magnitudes')
    plt.xlabel('Magnitude')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_PATH, 'magnitude_distribution.png'))
    plt.show()
else:
    print('Magnitude column not found.')

# Depth distribution
if 'depth' in df.columns:
    plt.figure(figsize=(10,4))
    plt.hist(df['depth'].dropna(), bins=30)
    plt.title('Distribution of Earthquake Depths')
    plt.xlabel('Depth (km)')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_PATH, 'depth_distribution.png'))
    plt.show()
else:
    print('Depth column not found.')


In [ ]:
# Boxplots for magnitude and depth by tsunami flag
if 'tsunami' in df.columns:
    if 'mag' in df.columns:
        data = [df.loc[df['tsunami'] == t, 'mag'].dropna() for t in sorted(df['tsunami'].unique())]
        labels = [str(t) for t in sorted(df['tsunami'].unique())]
        plt.figure(figsize=(8,5))
        plt.boxplot(data, labels=labels, showfliers=False)
        plt.title('Magnitude by Tsunami Flag (0 = No, 1 = Yes)')
        plt.xlabel('Tsunami')
        plt.ylabel('Magnitude')
        plt.tight_layout()
        plt.savefig(os.path.join(VIS_PATH, 'boxplot_mag_by_tsunami.png'))
        plt.show()

    if 'depth' in df.columns:
        data = [df.loc[df['tsunami'] == t, 'depth'].dropna() for t in sorted(df['tsunami'].unique())]
        labels = [str(t) for t in sorted(df['tsunami'].unique())]
        plt.figure(figsize=(8,5))
        plt.boxplot(data, labels=labels, showfliers=False)
        plt.title('Depth by Tsunami Flag (0 = No, 1 = Yes)')
        plt.xlabel('Tsunami')
        plt.ylabel('Depth (km)')
        plt.tight_layout()
        plt.savefig(os.path.join(VIS_PATH, 'boxplot_depth_by_tsunami.png'))
        plt.show()
else:
    print('Tsunami column not found; skipping boxplots by tsunami flag.')


In [ ]:
# Major earthquakes (>= 8.0)
if 'mag' in df.columns:
    major = df[df['mag'] >= 8.0]
    print('Number of major earthquakes (>= 8.0):', len(major))
    if len(major) > 0:
        display(major[['time','place','mag','depth','tsunami']].sort_values('mag', ascending=False))
else:
    print('Magnitude column not found; cannot list major earthquakes.')


In [ ]:
# Geographic scatter plot
if {'latitude', 'longitude'}.issubset(df.columns):
    plt.figure(figsize=(10,6))
    if 'tsunami' in df.columns:
        ts = df.dropna(subset=['latitude','longitude'])
        colors = ts['tsunami'].astype(float)
        plt.scatter(ts['longitude'], ts['latitude'], s=8, c=colors, cmap='viridis', alpha=0.6)
        plt.colorbar(label='Tsunami (0 = No, 1 = Yes)')
    else:
        plt.scatter(df['longitude'], df['latitude'], s=8, alpha=0.6)
    plt.title('Earthquake Locations (Longitude vs Latitude)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_PATH, 'geo_distribution.png'))
    plt.show()
else:
    print('Latitude/Longitude columns not found; skipping geographic plot.')


In [ ]:
# Correlation heatmap for numeric features
num_cols = [c for c in ['mag','depth','latitude','longitude','year'] if c in df.columns]
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    plt.figure(figsize=(6,5))
    plt.imshow(corr, interpolation='nearest', cmap='coolwarm')
    plt.xticks(range(len(num_cols)), num_cols, rotation=45)
    plt.yticks(range(len(num_cols)), num_cols)
    for i in range(len(num_cols)):
        for j in range(len(num_cols)):
            plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha='center', va='center', color='black')
    plt.title('Correlation matrix')
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_PATH, 'correlation_heatmap.png'))
    plt.show()
else:
    print('Not enough numeric columns for correlation heatmap.')


## Summary & Next Steps

- Visuals are saved to `/mnt/data/visuals/`.
- If you want a PDF report, we can programmatically export the notebook to PDF or use the saved images to build a report.
- Suggestions:
  - Clean missing values (impute or drop) depending on analysis needs.
  - Add geographic maps (folium) for interactive exploration.
  - Add model to predict tsunami likelihood (classification) using features like magnitude, depth, location.

---

**If you want, I can also:**
- Export these visuals into a PDF report.
- Package the whole project as a ZIP with the notebook, visuals, and README.
